# Notebook 2 — Acquiring Real-Time News with Playwright

**Goal:** scrape today's Google News *Top Stories*, follow the top 3
headlines to their publisher pages, and save the article text to
`data/articles/` for downstream question generation.

## Why scrape at all?

Every static benchmark is a snapshot: the moment it is published, models can
(and do) train on it, and the world it describes drifts away. If we want to
know whether a model is aware of *current* facts, we need test material the
model cannot have memorized — which means acquiring it live, at audit time.

News front pages are the perfect source: dense with verifiable, dated,
freshly minted facts.

## Why Playwright (and not `requests`)?

Google News is a JavaScript application — the raw HTML you get from a plain
HTTP GET contains almost none of the headlines. Playwright drives a real
(headless) Chromium browser, executes the page's JavaScript, and hands us the
rendered DOM. Two practical notes:

- Notebooks already run an asyncio event loop, so we use Playwright's
  **async API with top-level `await`** (the sync API raises inside Jupyter —
  this works the same in VS Code).
- Scraping live sites is inherently brittle: publishers block headless
  browsers, pages hang, markup changes. Every step below fails *gracefully*,
  and a fallback keeps the rest of the tutorial runnable even fully offline.


In [ ]:
import os
from pathlib import Path

# Notebooks live in notebooks/; run from the repo root so relative paths like
# ./data behave identically in VS Code and classic Jupyter. The local
# `toolkit` package is installed (editable) in the .venv, so no sys.path hacks.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

from toolkit.config import ARTICLES_DIR
from toolkit.playwright_helper import scrape_article_text

articles_dir = Path(ARTICLES_DIR)
articles_dir.mkdir(parents=True, exist_ok=True)
print(f"Articles will be saved to: {articles_dir.resolve()}")


## 1. Collect Top Stories links

Google News renders story links as **relative** anchors like
`./read/CBMi...`, so we:

1. select them with `a[href^="./read"]`,
2. convert to absolute URLs (`https://news.google.com/read/...`),
3. de-duplicate while preserving page order, keeping only anchors with
   headline-like text (the same story URL also appears on thumbnails and
   timestamps).


In [ ]:
from playwright.async_api import async_playwright

TOP_STORIES_URL = (
    "https://news.google.com/topics/"
    "CAAqJggKIiBDQkFTRWdvSUwyMHZNRFZxYUdjU0FtVnVHZ0pWVXlnQVAB"
)
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)"
)


async def collect_top_story_links(limit=10):
    links, seen = [], set()
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()
        try:
            await page.goto(TOP_STORIES_URL, wait_until="domcontentloaded", timeout=30000)
            await page.wait_for_timeout(3000)  # let the JS app render its feed
            for anchor in await page.query_selector_all('a[href^="./read"]'):
                href = await anchor.get_attribute("href")
                title = ((await anchor.inner_text()) or "").strip()
                # Keep only anchors with headline-like text; the same story
                # URL also appears on thumbnail and timestamp anchors.
                if not href or len(title) < 20:
                    continue
                absolute_url = "https://news.google.com" + href.lstrip(".")
                if absolute_url in seen:
                    continue
                seen.add(absolute_url)
                links.append({"title": title, "url": absolute_url})
                if len(links) >= limit:
                    break
        except Exception as e:
            print(f"Could not load Top Stories: {e}")
        finally:
            await browser.close()
    return links


story_links = await collect_top_story_links()
print(f"Found {len(story_links)} unique story links\n")
for i, link in enumerate(story_links[:5], 1):
    print(f"{i}. {link['title'][:90]}")


## 2. Scrape the top 3 articles

A Google News story URL is not the article — it is a tiny JavaScript shell
that redirects the browser to the publisher's site. If you scrape it
directly you capture ~11 characters of shell before the redirect fires. So
we scrape in two hops:

1. **Resolve**: open the story URL and wait until the browser's location
   leaves `news.google.com` — that final URL is the publisher page.
2. **Scrape**: pass the publisher URL to `scrape_article_text` (from
   `toolkit.playwright_helper`), which strips
   `script/style/nav/footer/header/aside` tags and returns cleaned text.

**Quality gate** — a result is replaced by a bundled fallback sample article
(clearly labeled) when:

- the redirect never resolved, or landed on `google.com/sorry` (Google's
  rate-limit/captcha interstitial — expected under bursty automated access,
  e.g. a room full of workshop participants on one conference network);
- the scrape errored (timeouts are common on live-blog pages that never
  reach `networkidle`); or
- fewer than 500 characters of text survived cleaning.

This keeps Notebooks 3–4 runnable in every environment, including fully
offline.


In [ ]:
FALLBACK_ARTICLES = [
    (
        "[FALLBACK SAMPLE ARTICLE - not live news]\n"
        "City Council Approves $2.4 Billion Transit Expansion After Marathon Session\n"
        "The Riverton City Council voted 8-3 late Tuesday to approve the Eastline "
        "light-rail expansion, a $2.4 billion project that will add 14 stations "
        "across 19 miles and connect the Fairmont district to the downtown core by "
        "2031. Council member Rosa Delgado, who chaired the transportation "
        "committee, called the vote 'the largest infrastructure commitment in the "
        "city's history.' The project is funded by a 0.5 percent sales tax approved "
        "by voters in November, federal grants covering 38 percent of costs, and "
        "municipal bonds. Opponents, led by council member Hal Brennan, argued the "
        "ridership projections of 62,000 daily boardings were inflated and pointed "
        "to the Westline's 40 percent shortfall against its own forecasts. "
        "Construction is scheduled to begin in March 2027, with the Fairmont "
        "segment opening first in late 2029. The transit authority estimates the "
        "project will create 6,800 construction jobs."
    ),
    (
        "[FALLBACK SAMPLE ARTICLE - not live news]\n"
        "Researchers Report Breakthrough in Long-Duration Grid Storage\n"
        "A team at Corvallis Polytechnic announced Wednesday that its iron-air "
        "battery prototype sustained 118 hours of continuous discharge, nearly "
        "five days, at a cost the team estimates at $18 per kilowatt-hour — about "
        "a tenth of typical lithium-ion storage costs. The results, published in "
        "the journal Energy Systems, describe a 40-kilowatt demonstration unit "
        "operated through 1,200 charge cycles with 9 percent capacity loss. Lead "
        "author Dr. Amara Okonkwo said the chemistry relies on reversible rusting "
        "of iron pellets and uses no cobalt, nickel, or lithium. Independent "
        "reviewers cautioned that round-trip efficiency remains low at 43 percent, "
        "meaning more than half the input energy is lost. Startup FerroGrid has "
        "licensed the design and plans a 5-megawatt pilot with the Cascade Valley "
        "utility district in 2028."
    ),
    (
        "[FALLBACK SAMPLE ARTICLE - not live news]\n"
        "National Parks Set Visitation Record as Reservation System Expands\n"
        "The National Park Service reported Thursday that the system logged 342 "
        "million recreational visits last year, a 4.2 percent increase and the "
        "highest total ever recorded. Zion, Arches, and Glacier again required "
        "timed-entry reservations during peak months, and the agency announced the "
        "program will expand to Rocky Mountain's Bear Lake corridor and parts of "
        "Acadia next summer. Park Service director June Calloway said crowding "
        "remains concentrated: 28 percent of all visits occurred in just 10 of the "
        "system's 433 units. The agency's $23 billion deferred-maintenance backlog "
        "drew criticism at a Senate oversight hearing, where Sen. Marcus Doyle "
        "questioned spending on new visitor centers while trail systems degrade. "
        "Entrance fee revenue rose to $412 million, of which 80 percent stays in "
        "the collecting park under current law."
    ),
]


async def resolve_publisher_url(gn_url, timeout_ms=20000):
    # Google News story pages are JS shells that redirect to the publisher;
    # wait for the browser location to leave news.google.com.
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=USER_AGENT)
        page = await context.new_page()
        try:
            await page.goto(gn_url, wait_until="domcontentloaded", timeout=timeout_ms)
            try:
                await page.wait_for_url(lambda u: "news.google.com" not in u, timeout=timeout_ms)
            except Exception:
                pass  # never redirected; caller's quality gate handles it
            return page.url
        except Exception:
            return gn_url
        finally:
            await browser.close()


N_ARTICLES = 3
MIN_CHARS = 500

saved = []
for i in range(N_ARTICLES):
    out_path = articles_dir / f"article_{i + 1}.txt"
    text, source = "", "fallback"
    if i < len(story_links):
        print(f"Article {i + 1}: {story_links[i]['title'][:70]}")
        publisher_url = await resolve_publisher_url(story_links[i]["url"])
        if "news.google.com" in publisher_url or "google.com/sorry" in publisher_url:
            print(f"  -> redirect blocked/unresolved ({publisher_url[:60]}...), using fallback sample")
        else:
            print(f"  -> publisher: {publisher_url[:70]}")
            text = await scrape_article_text(publisher_url)
            if not text.startswith("Failed scraping URL") and len(text) >= MIN_CHARS:
                source = "live"
            else:
                print(f"  -> scrape unusable ({text[:80]!r}...), using fallback sample")
    else:
        print(f"Article {i + 1}: no live link available, using fallback sample")
    if source == "fallback":
        text = FALLBACK_ARTICLES[i]
    out_path.write_text(text, encoding="utf-8")
    saved.append({"file": out_path.name, "source": source, "chars": len(text)})
    print(f"  saved {out_path} ({len(text):,} chars, source={source})")

import pandas as pd
pd.DataFrame(saved)


In [ ]:
# Preview what we captured
for i in range(1, N_ARTICLES + 1):
    path = articles_dir / f"article_{i}.txt"
    text = path.read_text(encoding="utf-8")
    print("=" * 78)
    print(f"{path.name} — {len(text):,} chars")
    print("=" * 78)
    print(text[:600])
    print("...\n")


## Takeaways

- Real-time auditing starts with **real-time acquisition** — a rendered-DOM
  scraper is table stakes for JS-heavy sources like Google News.
- Extraction is noisy (cookie banners, related-story links survive tag
  stripping). For MCQ generation that is acceptable; production pipelines add
  boilerplate-removal models.
- Graceful degradation matters: a live demo that dies on a blocked scrape
  teaches nothing. Check the `source` column above to see which articles are
  live vs. fallback.

**Next:** Notebook 3 turns these articles into strict-JSON multiple-choice
questions — our synthetic, annotator-free test set.
